In [1]:
import os
import zipfile
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd   

from PIL import Image
from tqdm import tqdm 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
zip_path = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"
print("Exists:", os.path.exists(zip_path))

Exists: True


In [5]:
extract_dir = "/content/dataset"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Unzipped to:", extract_dir)

Unzipped to: /content/dataset


In [6]:
for root, dirs, files in os.walk(extract_dir):
    if "scenario23" in root.lower():
        print(root)

/content/dataset/scenario23_dev
/content/dataset/scenario23_dev/resources
/content/dataset/scenario23_dev/resources/bbox_labels_final
/content/dataset/scenario23_dev/unit2
/content/dataset/scenario23_dev/unit2/roll
/content/dataset/scenario23_dev/unit2/GPS_data
/content/dataset/scenario23_dev/unit2/altitude
/content/dataset/scenario23_dev/unit2/y_speed
/content/dataset/scenario23_dev/unit2/distance
/content/dataset/scenario23_dev/unit2/speed
/content/dataset/scenario23_dev/unit2/pitch
/content/dataset/scenario23_dev/unit2/z_speed
/content/dataset/scenario23_dev/unit2/x_speed
/content/dataset/scenario23_dev/unit2/height
/content/dataset/scenario23_dev/unit1
/content/dataset/scenario23_dev/unit1/GPS_data
/content/dataset/scenario23_dev/unit1/camera_data
/content/dataset/scenario23_dev/unit1/mmWave_data


In [7]:
author_dir = "/content/Image beam"
os.makedirs(author_dir, exist_ok=True)


print(author_dir)

/content/Image beam


In [8]:
AUTHOR_ROOT = "/content/drive/MyDrive/Image beam"

In [9]:
DATA_ROOT = "/content/dataset/scenario23_dev"   # update if needed
print("Exists:", os.path.exists(DATA_ROOT))

Exists: True


In [10]:
AUTHOR_ROOT = "/content/drive/MyDrive/Image beam"  # adjust path

needed = [
    "build_net.py",
    "data_feed.py",
    "main_beam.py",
    "main_beam_eval.py",
    "scenario23_img_beam_train.csv",
    "scenario23_img_beam_val.csv",
    "scenario23_img_beam_test.csv",
]

for f in needed:
    p = os.path.join(AUTHOR_ROOT, f)
    print(f, "->", os.path.exists(p), p)

build_net.py -> True /content/drive/MyDrive/Image beam/build_net.py
data_feed.py -> True /content/drive/MyDrive/Image beam/data_feed.py
main_beam.py -> True /content/drive/MyDrive/Image beam/main_beam.py
main_beam_eval.py -> True /content/drive/MyDrive/Image beam/main_beam_eval.py
scenario23_img_beam_train.csv -> True /content/drive/MyDrive/Image beam/scenario23_img_beam_train.csv
scenario23_img_beam_val.csv -> True /content/drive/MyDrive/Image beam/scenario23_img_beam_val.csv
scenario23_img_beam_test.csv -> True /content/drive/MyDrive/Image beam/scenario23_img_beam_test.csv


In [11]:
train_csv = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_train.csv")
df_img_author = pd.read_csv(train_csv)

print(df_img_author.head())
print(df_img_author.columns.tolist())

   index                                          unit1_rgb  unit1_beam
0   3532  ../scenario23_dev/unit1/camera_data/image_BS1_...          17
1   2224  ../scenario23_dev/unit1/camera_data/image_BS1_...          14
2   9416  ../scenario23_dev/unit1/camera_data/image_BS1_...          17
3   8510  ../scenario23_dev/unit1/camera_data/image_BS1_...          20
4   6877  ../scenario23_dev/unit1/camera_data/image_BS1_...          17
['index', 'unit1_rgb', 'unit1_beam']


In [12]:
sample_paths = df_img_author.iloc[:10, 1].tolist()
print(sample_paths)

for p in sample_paths[:5]:
    print("RAW:", p, "EXISTS:", os.path.exists(str(p)))

['../scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_2258_17_04_41.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_2121_17_04_20.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_11899_18_02_04.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_8210_17_50_08.jpg']
RAW: ../scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg EXISTS: False
RAW: ../scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg EXISTS: False
RAW: ../scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg EXISTS: False
RAW: ../scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg EXISTS: 

In [13]:
from pathlib import Path

image_map = {}
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            image_map[f] = os.path.join(root, f)

print("Indexed images:", len(image_map))
print(list(image_map.items())[:5])

Indexed images: 11387
[('image_BS1_10669_17_57_59.jpg', '/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10669_17_57_59.jpg'), ('image_BS1_6108_17_44_33.jpg', '/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_6108_17_44_33.jpg'), ('image_BS1_11714_18_01_16.jpg', '/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_11714_18_01_16.jpg'), ('image_BS1_5118_17_14_04.jpg', '/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_5118_17_14_04.jpg'), ('image_BS1_5621_17_15_24.jpg', '/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_5621_17_15_24.jpg')]


In [14]:
def fix_img_csv_paths(csv_path, out_path):
    dfc = pd.read_csv(csv_path).copy()

    # author code takes row.values[1:3], so column index 1 must be image path
    path_col = dfc.columns[1]

    dfc[path_col] = dfc[path_col].apply(lambda x: image_map.get(os.path.basename(str(x).strip()), str(x)))
    dfc.to_csv(out_path, index=False)

    print("Saved:", out_path)
    print(dfc.head())
    return dfc

fixed_train = fix_img_csv_paths(
    os.path.join(AUTHOR_ROOT, "scenario23_img_beam_train.csv"),
    "/content/scenario23_img_beam_train_fixed.csv"
)

fixed_val = fix_img_csv_paths(
    os.path.join(AUTHOR_ROOT, "scenario23_img_beam_val.csv"),
    "/content/scenario23_img_beam_val_fixed.csv"
)

fixed_test = fix_img_csv_paths(
    os.path.join(AUTHOR_ROOT, "scenario23_img_beam_test.csv"),
    "/content/scenario23_img_beam_test_fixed.csv"
)

Saved: /content/scenario23_img_beam_train_fixed.csv
   index                                          unit1_rgb  unit1_beam
0   3532  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   2224  /content/dataset/scenario23_dev/unit1/camera_d...          14
2   9416  /content/dataset/scenario23_dev/unit1/camera_d...          17
3   8510  /content/dataset/scenario23_dev/unit1/camera_d...          20
4   6877  /content/dataset/scenario23_dev/unit1/camera_d...          17
Saved: /content/scenario23_img_beam_val_fixed.csv
   index                                          unit1_rgb  unit1_beam
0   1542  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   6743  /content/dataset/scenario23_dev/unit1/camera_d...          10
2  10999  /content/dataset/scenario23_dev/unit1/camera_d...           2
3   5992  /content/dataset/scenario23_dev/unit1/camera_d...          15
4   7698  /content/dataset/scenario23_dev/unit1/camera_d...           7
Saved: /content/scenario23_img_bea

In [15]:
for p in fixed_train.iloc[:10, 1].tolist():
    print(p, os.path.exists(str(p)))

/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2258_17_04_41.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2121_17_04_20.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_11899_18_02_04.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_8210_17_50_08.jpg True


In [16]:
from skimage import io
from torch.utils.data import Dataset, DataLoader


def create_samples(root):
    f = pd.read_csv(root)
    data_samples = []
    for idx, row in f.iterrows():
        img_paths = row.values[1:3]
        data_samples.append(img_paths)
    return data_samples

class DataFeedAuthor(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root = root_dir
        self.samples = create_samples(self.root)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = io.imread(sample[0])
        if self.transform:
            img = self.transform(img)
        label = int(sample[1])
        return img, label

In [17]:
import torchvision.transforms as transf

img_resize = transf.Resize((224, 224))
img_norm = transf.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))

proc_pipe = transf.Compose([
    transf.ToPILImage(),
    img_resize,
    transf.ToTensor(),
    img_norm
])

In [18]:
batch_size = 64
val_batch_size = 64

train_loader_img = DataLoader(
    DataFeedAuthor("/content/scenario23_img_beam_train_fixed.csv", transform=proc_pipe),
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

val_loader_img = DataLoader(
    DataFeedAuthor("/content/scenario23_img_beam_val_fixed.csv", transform=proc_pipe),
    batch_size=val_batch_size,
    shuffle=True, num_workers=2
)

test_loader_img = DataLoader(
    DataFeedAuthor("/content/scenario23_img_beam_test_fixed.csv", transform=proc_pipe),
    batch_size=val_batch_size,
    shuffle=True, num_workers=2
)

In [19]:
imgs, labels = next(iter(train_loader_img))
print(imgs.shape, labels.shape)
print(labels[:10])

torch.Size([64, 3, 224, 224]) torch.Size([64])
tensor([17,  4,  2, 26, 15, 20, 15, 30, 11, 17])


In [20]:

from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model_img = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model_img.fc = nn.Linear(model_img.fc.in_features, 64)   # author code uses 64
model_img = model_img.to(device)

print(model_img.fc)

Device: cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 205MB/s]


Linear(in_features=2048, out_features=64, bias=True)


In [21]:
def evaluate_topk(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()

    total = 0
    correct = {k: 0 for k in ks}

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)
            max_k = max(ks)
            _, pred = torch.topk(outputs, k=max_k, dim=1)
            pred = pred.t()

            total += labels.size(0)

            for k in ks:
                correct_k = pred[:k].eq(labels.view(1, -1)).sum().item()
                correct[k] += correct_k

    return {f"top{k}": 100.0 * correct[k] / total for k in ks}

In [22]:
import torch.optim as optim


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_img.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[4, 8, 12], gamma=0.1)

num_epochs = 20
best_top1 = -1
best_state = None
history = []

for epoch in range(1, num_epochs + 1):
    model_img.train()
    running_loss = 0.0
    total_train = 0

    for imgs, labels in train_loader_img:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model_img(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        bs = labels.size(0)
        running_loss += loss.item() * bs
        total_train += bs

    scheduler.step()

    train_loss = running_loss / total_train
    val_metrics = evaluate_topk(model_img, val_loader_img, device, ks=(1, 2, 3, 5))

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Top1: {val_metrics['top1']:.2f} | "
        f"Top2: {val_metrics['top2']:.2f} | "
        f"Top3: {val_metrics['top3']:.2f} | "
        f"Top5: {val_metrics['top5']:.2f}"
    )

    if val_metrics["top1"] > best_top1:
        best_top1 = val_metrics["top1"]
        best_state = {k: v.detach().cpu().clone() for k, v in model_img.state_dict().items()}
        torch.save(best_state, "/content/best_resnet50_img.pth")
        print("Saved best model")

Epoch 01 | Train Loss: 0.9755 | Top1: 81.18 | Top2: 95.93 | Top3: 98.16 | Top5: 99.30
Saved best model
Epoch 02 | Train Loss: 0.4558 | Top1: 84.22 | Top2: 97.42 | Top3: 99.15 | Top5: 99.47
Saved best model
Epoch 03 | Train Loss: 0.4227 | Top1: 86.42 | Top2: 97.72 | Top3: 99.21 | Top5: 99.50
Saved best model
Epoch 04 | Train Loss: 0.4031 | Top1: 85.71 | Top2: 97.63 | Top3: 99.33 | Top5: 99.56
Epoch 05 | Train Loss: 0.2957 | Top1: 87.50 | Top2: 98.45 | Top3: 99.39 | Top5: 99.59
Saved best model
Epoch 06 | Train Loss: 0.2627 | Top1: 87.18 | Top2: 98.24 | Top3: 99.36 | Top5: 99.68
Epoch 07 | Train Loss: 0.2438 | Top1: 87.76 | Top2: 98.33 | Top3: 99.33 | Top5: 99.68
Saved best model
Epoch 08 | Train Loss: 0.2221 | Top1: 87.27 | Top2: 98.13 | Top3: 99.27 | Top5: 99.65
Epoch 09 | Train Loss: 0.1894 | Top1: 87.59 | Top2: 98.33 | Top3: 99.27 | Top5: 99.65
Epoch 10 | Train Loss: 0.1792 | Top1: 87.79 | Top2: 98.24 | Top3: 99.24 | Top5: 99.65
Saved best model
Epoch 11 | Train Loss: 0.1729 | Top1: 

In [23]:
hist_img = pd.DataFrame(history)
hist_img

,epoch,train_loss,top1,top2,top3,top5
0,1,0.975490,81.176815,95.930913,98.155738,99.297424
1,2,0.455783,84.221311,97.423888,99.151054,99.473068
2,3,0.422650,86.416862,97.716628,99.209602,99.502342
3,4,0.403070,85.714286,97.628806,99.326698,99.560890
4,5,0.295738,87.500000,98.448478,99.385246,99.590164
5,6,0.262680,87.177986,98.243560,99.355972,99.677986
6,7,0.243764,87.763466,98.331382,99.326698,99.677986
7,8,0.222142,87.265808,98.126464,99.268150,99.648712
8,9,0.189439,87.587822,98.331382,99.268150,99.648712
9,10,0.179237,87.792740,98.243560,99.238876,99.648712


In [24]:
best_state = torch.load("/content/best_resnet50_img.pth", map_location=device)
model_img.load_state_dict(best_state)
model_img = model_img.to(device)

test_metrics = evaluate_topk(model_img, test_loader_img, device, ks=(1, 2, 3, 5))
print("Test metrics:", test_metrics)

Test metrics: {'top1': 88.23529411764706, 'top2': 98.06848112379281, 'top3': 99.64881474978051, 'top5': 99.91220368744513}


In [26]:
results_img = pd.DataFrame([{
    "Modality": "Image (ResNet50)",
    "Top-1": test_metrics["top1"],
    "Top-2": test_metrics["top2"],
    "Top-3": test_metrics["top3"],
    "Top-5": test_metrics["top5"],
}])

results_img

,Modality,Top-1,Top-2,Top-3,Top-5
0,Image (ResNet50),88.235294,98.068481,99.648815,99.912204


mlp cell 

In [27]:
AUTHOR_POS_ROOT = "/content/drive/MyDrive/Pos beam"   # change path if needed

In [28]:
pos_train_csv = os.path.join(AUTHOR_POS_ROOT, "scenario23_pos_beam_train.csv")
pos_val_csv   = os.path.join(AUTHOR_POS_ROOT, "scenario23_pos_beam_val.csv")
pos_test_csv  = os.path.join(AUTHOR_POS_ROOT, "scenario23_pos_beam_test.csv")

df_pos_author = pd.read_csv(pos_train_csv)
print(df_pos_author.head())
print(df_pos_author.columns.tolist())

   index                                  unit2_pos  unit1_beam
0   3532    [0.8092883966431671, 0.521083920903955]          17
1   2224  [0.4816276084988933, 0.29434536152734486]          14
2   9416    [0.220278556834608, 0.4136596156292844]          17
3   8510  [0.21412273613497904, 0.4547214157104936]          20
4   6877  [0.14500641727379412, 0.4097884695072434]          17
['index', 'unit2_pos', 'unit1_beam']


In [29]:
import ast
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader

def create_pos_samples(root):
    f = pd.read_csv(root)
    data_samples = []
    for _, row in f.iterrows():
        data = list(row.values[1:])
        data_samples.append(data)
    return data_samples

class DataFeedPosAuthor(Dataset):
    def __init__(self, root_dir):
        self.root = root_dir
        self.samples = create_pos_samples(self.root)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        pos_val = sample[:1]
        pos_val = ast.literal_eval(pos_val[0])
        pos_val = np.asarray(pos_val, dtype=np.float32)

        pos_centers = sample[1:2]
        pos_centers = np.asarray(pos_centers, dtype=np.int64)

        return pos_val, pos_centers

In [43]:
batch_size = 128
val_batch_size = 64

train_loader_pos_author = DataLoader(
    DataFeedPosAuthor(pos_train_csv),
    batch_size=batch_size,
    shuffle=False
)

val_loader_pos_author = DataLoader(
    DataFeedPosAuthor(pos_val_csv),
    batch_size=val_batch_size,
    shuffle=False 
)

test_loader_pos_author = DataLoader(
    DataFeedPosAuthor(pos_test_csv),
    batch_size=val_batch_size,
    shuffle=False
)

In [44]:
pos_batch, label_batch = next(iter(train_loader_pos_author))
print(pos_batch.shape, label_batch.shape)
print(pos_batch[:3])
print(label_batch[:10])

torch.Size([128, 2]) torch.Size([128, 1])
tensor([[0.8093, 0.5211],
        [0.4816, 0.2943],
        [0.2203, 0.4137]])
tensor([[17],
        [14],
        [17],
        [20],
        [17],
        [17],
        [12],
        [ 9],
        [17],
        [12]])


In [45]:
input_size = 2
node = 512
output_size = 65

class NN_beam_pred(nn.Module):
    def __init__(self, num_features, num_output):
        super(NN_beam_pred, self).__init__()
        self.layer_1 = nn.Linear(num_features, node)
        self.layer_2 = nn.Linear(node, node)
        self.layer_3 = nn.Linear(node, node)
        self.layer_out = nn.Linear(node, num_output)
        self.relu = nn.ReLU()

    def forward(self, inputs):
        x = self.relu(self.layer_1(inputs))
        x = self.relu(self.layer_2(x))
        x = self.relu(self.layer_3(x))
        x = self.layer_out(x)
        return x

model_pos_author = NN_beam_pred(input_size, output_size).to(device)
print(model_pos_author)

NN_beam_pred(
  (layer_1): Linear(in_features=2, out_features=512, bias=True)
  (layer_2): Linear(in_features=512, out_features=512, bias=True)
  (layer_3): Linear(in_features=512, out_features=512, bias=True)
  (layer_out): Linear(in_features=512, out_features=65, bias=True)
  (relu): ReLU()
)


In [46]:
def evaluate_topk_pos(model, loader, device, ks=(1, 2, 3)):
    model.eval()

    total = 0
    correct = {k: 0 for k in ks}

    with torch.no_grad():
        for pos_data, beam_val in loader:
            x = pos_data.float().to(device)
            labels = beam_val[:, 0].long().to(device)

            out = model(x)

            max_k = max(ks)
            _, pred = torch.topk(out, k=max_k, dim=1)
            pred = pred.t()

            total += labels.size(0)

            for k in ks:
                correct_k = pred[:k].eq(labels.view(1, -1)).sum().item()
                correct[k] += correct_k

    return {f"top{k}": 100.0 * correct[k] / total for k in ks}

In [49]:
import torch.optim as optim


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_pos_author.parameters(), lr=0.01, weight_decay=1e-4)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[15, 25, 40], gamma=0.1)

num_epochs = 50
best_top1 = -1
best_state = None
history_pos = []

for epoch in range(1, num_epochs + 1):
    model_pos_author.train()
    running_loss = 0.0
    total_train = 0

    for pos_data, beam_val in train_loader_pos_author:
        x = pos_data.float().to(device)
        labels = beam_val[:, 0].long().to(device)

        optimizer.zero_grad()
        out = model_pos_author(x)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()

        bs = labels.size(0)
        running_loss += loss.item() * bs
        total_train += bs

    scheduler.step()

    train_loss = running_loss / total_train
    val_metrics = evaluate_topk_pos(model_pos_author, val_loader_pos_author, device, ks=(1, 2, 3))

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics
    }
    history_pos.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Top1: {val_metrics['top1']:.2f} | "
        f"Top2: {val_metrics['top2']:.2f} | "
        f"Top3: {val_metrics['top3']:.2f}"
    )

    if val_metrics["top1"] > best_top1:
        best_top1 = val_metrics["top1"]
        best_state = {k: v.detach().cpu().clone() for k, v in model_pos_author.state_dict().items()}
        torch.save(best_state, "/content/best_pos_author.pth")
        print("Saved best position model")

Epoch 01 | Train Loss: 1.2530 | Top1: 55.97 | Top2: 79.13 | Top3: 89.17
Saved best position model
Epoch 02 | Train Loss: 1.1615 | Top1: 56.12 | Top2: 78.89 | Top3: 89.23
Saved best position model
Epoch 03 | Train Loss: 1.1505 | Top1: 56.50 | Top2: 79.51 | Top3: 90.02
Saved best position model
Epoch 04 | Train Loss: 1.1430 | Top1: 57.82 | Top2: 80.24 | Top3: 89.96
Saved best position model
Epoch 05 | Train Loss: 1.1272 | Top1: 58.28 | Top2: 80.56 | Top3: 90.40
Saved best position model
Epoch 06 | Train Loss: 1.1254 | Top1: 57.90 | Top2: 79.89 | Top3: 90.49
Epoch 07 | Train Loss: 1.1225 | Top1: 57.06 | Top2: 79.83 | Top3: 89.78
Epoch 08 | Train Loss: 1.1218 | Top1: 57.85 | Top2: 80.77 | Top3: 90.05
Epoch 09 | Train Loss: 1.1181 | Top1: 58.81 | Top2: 81.03 | Top3: 91.07
Saved best position model
Epoch 10 | Train Loss: 1.1119 | Top1: 57.73 | Top2: 80.47 | Top3: 90.16
Epoch 11 | Train Loss: 1.1165 | Top1: 59.13 | Top2: 80.77 | Top3: 90.84
Saved best position model
Epoch 12 | Train Loss: 1.1

In [50]:
best_state = torch.load("/content/best_pos_author.pth", map_location=device)
model_pos_author.load_state_dict(best_state)
model_pos_author = model_pos_author.to(device)

test_metrics_pos = evaluate_topk_pos(model_pos_author, test_loader_pos_author, device, ks=(1, 2, 3))
print("Position Test metrics:", test_metrics_pos)

Position Test metrics: {'top1': 59.43810359964881, 'top2': 83.40649692712906, 'top3': 92.44951712028094}
